# Wordle strategy explorer

Explore the **optimal trees** — the committed per-opener strategies the solver emits — as a
table of *positions*. `TOP_K` picks how many of them to explore, best average guesses first
(`TOP_K = 1` is `salet`, the optimum), and the cells below enumerate every situation those
strategies can reach: the guesses played so far, the feedback each one got, how many answers
are still possible, and the optimal next guess. Above `TOP_K = 1` the trees are POOLED, each
weighted equally, and every row carries the `opener` it came from.

Modelled on `hangman/math/notebooks/strategy_explorer.ipynb` ("All situations at depth k",
"Inspect a tree node"), over Wordle's tree files instead of hangman's.

**Where the data comes from.** `paths.TREES` resolves to the tree directory this machine
should read (the live `local/<solver>/trees/` on a sweep machine, else the committed
`shared/<solver>/trees/`). One gzipped file per opener, one line per answer:

```
nymph 3 salet bbbbb courd bbbbb nymph
```

— the answer, its guess count, then `guess pattern guess pattern ... guess`. There are N
guesses and N-1 patterns, because the last guess *is* the answer and its feedback is
all-green by construction. A pattern is a 5-character string: **`b`** = grey, **`y`** =
yellow, **`g`** = green.

`math/notebooks/opener_distribution.ipynb` is the companion that plots every opener's
optimal average across the whole sweep, rather than walking a handful of trees.

Setting `WORDLE_SOLVER=hardturns` (or `ansturns`) before starting the kernel points every
cell at that sweep instead.

> **The objective these trees optimise is CONSTRAINED.** Each tree minimises the TOTAL
> guesses over the 2315 answers *subject to resolving every answer within `--depth`*. That
> is not the same as minimum expected guesses, and anything read off these positions
> inherits that constraint.

In [1]:
from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'wordle.py').exists():
            return c
    raise FileNotFoundError('Could not find wordle.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
for p in (math_dir, math_dir / 'tools'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import wordle                     # the feedback primitives (math/wordle.py)
import paths                      # the canonical solver locations (math/tools/paths.py)

pd.set_option('display.max_rows', 200)   # show up to 200 rows before pandas truncates to a head/tail preview

print(f'imported wordle + paths from {math_dir}')
print(f'solver = {paths.SOLVER!r}')
print(f'trees  = {paths.TREES}')

imported wordle + paths from /Users/patrickliscio/Documents/Ballpark Figures/ballpark-figures/wordle/math
solver = 'optturns'
trees  = /Users/patrickliscio/Documents/Ballpark Figures/ballpark-figures/wordle/math/data/shared/optturns/trees


## Pick the trees

`TOP_K` takes the best openers by **average guesses** — the committed `total` over the
2,315 answers, ties broken on the word so the pick is deterministic. `TOP_K = 1` is just
`salet`, the optimum, and every cell below then reads exactly the one tree it always did.
Above 1 the cells POOL those trees: each row of every table carries the `opener` whose tree
it came from, and each tree contributes its own 2,315 answers, so **every opener in the
selection carries equal weight however good it is** — the same convention
`tools/tree_states.py --top N` uses.

`OPENERS` names trees explicitly instead (`['salet', 'crate']`), for a comparison the
ranking would not produce.

The trees are the **committed copy**, which can lag the sweep's CSV, so an opener ranked
inside the top K with no tree on disk is reported and skipped rather than silently dropped.

In [ ]:
TOP_K   = 1        # <-- how many trees to explore, best average first (1 = just salet)
OPENERS = None     # <-- or name them: ['salet', 'crate']; overrides TOP_K when set


def select_openers(top_k=None, openers=None):
    """The openers to explore, and the totals frame describing them.

    `openers` is taken verbatim; otherwise the `top_k` best by the committed
    `total` (ties on the word, so the pick is deterministic). Either way the
    result is restricted to openers with a tree file on disk -- `paths.TREES` can
    be a committed snapshot that lags the CSV -- and anything dropped for a
    missing tree is named rather than silently skipped."""
    totals = paths.load_totals()
    have = {p.name[:-len('.txt.gz')] for p in Path(paths.TREES).glob('*.txt.gz')}

    if openers:
        picked = [w for w in openers if w in have]
        skipped = [w for w in openers if w not in have]
    else:
        picked, skipped = [], []
        for w in sorted(totals, key=lambda w: (totals[w], w)):
            if len(picked) == top_k:
                break
            (picked if w in have else skipped).append(w)

    if skipped:
        print(f'no tree in {paths.TREES} for: {" ".join(skipped)} -- skipped')
    if not picked:
        raise FileNotFoundError('no trees to read: every requested opener is missing one')

    frame = pd.DataFrame({'opener': picked,
                          'total': [totals.get(w) for w in picked]})
    frame['expected'] = frame['total'] / paths.N_ANSWERS
    return picked, frame


TREE_OPENERS, tree_totals = select_openers(TOP_K, OPENERS)
LABEL = (TREE_OPENERS[0] if len(TREE_OPENERS) == 1
         else f'top {len(TREE_OPENERS)}: ' + ' '.join(TREE_OPENERS[:4])
              + (' ...' if len(TREE_OPENERS) > 4 else ''))

print(f'{len(TREE_OPENERS)} tree(s): {LABEL}')
print(f'total {tree_totals["total"].min():,} .. {tree_totals["total"].max():,} guesses over '
      f'{paths.N_ANSWERS:,} answers each '
      f'({tree_totals["expected"].min():.5f} .. {tree_totals["expected"].max():.5f} per answer)')
display(tree_totals.head(20))

## Every position in the tree

`positions(openers)` returns one row per **position**: a distinct history of
(guess, feedback) pairs one of the selected strategies can reach.

| column | meaning |
| --- | --- |
| `opener` | which selected tree this position belongs to |
| `depth` | guesses already played (0 = the root, before the opener) |
| `guesses` | those guesses, in order, space separated |
| `patterns` | the feedback each one got, space separated `byg` strings |
| `n_answers` | answers still possible at this position |
| `guess` | the **optimal next guess** — the tree's move here |
| `candidate` | that guess is itself one of the still-possible answers, so it *could* win outright; `False` makes it a **probe**, played for information only |
| `solved` | one answer left, so `guess` just plays it |
| `sample` | a few of the possible answers, for identification |

**How it is derived.** Every answer's line records the full path the strategy walks for
that answer, so taking each *prefix* of its (guess, pattern) pairs enumerates the positions
that answer passes through. Answers sharing a prefix are exactly the answers still
consistent with that history — the strategy is deterministic, so two answers part company
only where their feedback differs — which is why counting the lines through a node gives
its candidate count. The last row of a path is the position where one answer remains and
`guess` is that answer.

`candidate` is that same set read once more: the guess is compatible iff it is one of the
answers still alive here, which is exactly the answers on the lines through this node. So a
guess-only word — 10,657 of the 12,972 legal words are not mystery words, `salet` and
`courd` among them — is never a candidate at any position, and a `solved` position is always
one (it plays the last answer standing).

Exactly **2,315 positions per tree** come back a candidate, and that is an identity rather
than a coincidence: a compatible guess is itself an answer, and being alive at the position
means the position sits on that answer's own path — which plays it exactly once, since the
reply would otherwise be all green and the game over. So candidate positions pair one to one
with answers, which makes the printed count a free check that the parse is sound.

Positions are keyed **per tree**: two openers that reach the same history are two rows, and
the same word played by two trees is counted once for each. With `TOP_K = 1` there is one
tree and the frame is exactly the single-opener one.

In [ ]:
def read_tree(opener, trees=None):
    """Parse one committed tree; yield (answer, n_guesses, guesses, patterns) per answer.

    `guesses` has n entries and `patterns` has n-1: the final guess is the answer
    itself and carries no pattern (its feedback is all-green by construction)."""
    path = Path(trees or paths.TREES) / f'{opener}.txt.gz'
    if not path.exists():
        raise FileNotFoundError(f'no tree for {opener!r} at {path}')
    with gzip.open(path, 'rt') as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            yield parts[0], int(parts[1]), parts[2::2], parts[3::2]


def positions(openers, trees=None, sample=4):
    """Every position reachable under each of `openers`' optimal trees.

    One row per (opener, history) -- see the markdown above for the columns and
    how the counts are derived. Trees are pooled, never merged: positions are
    keyed per opener, so each contributes its own 2315 answers and the selection
    weights every tree equally. `sample` caps how many possible answers each row
    lists."""
    if isinstance(openers, str):
        openers = [openers]

    rows = []
    for opener in openers:
        nodes = {}
        for answer, n, guesses, pats in read_tree(opener, trees):
            for k in range(len(guesses)):        # k pairs played, guesses[k] is next
                key = tuple(zip(guesses[:k], pats[:k]))
                node = nodes.get(key)
                if node is None:
                    nodes[key] = node = {'guess': guesses[k], 'answers': []}
                elif node['guess'] != guesses[k]:
                    # A tree is a deterministic strategy: one move per position. Two
                    # different moves after the same history means it is not one.
                    raise ValueError(f'{opener}: position {key} has two moves, '
                                     f'{node["guess"]!r} and {guesses[k]!r}')
                node['answers'].append(answer)

        for key, node in nodes.items():
            answers = node['answers']
            rows.append({
                'opener': opener,
                'depth': len(key),
                'guesses': ' '.join(g for g, _ in key),
                'patterns': ' '.join(p for _, p in key),
                'n_answers': len(answers),
                'guess': node['guess'],
                # The answers on the lines through this node ARE the answers still
                # consistent with its history, so membership is compatibility.
                'candidate': node['guess'] in set(answers),
                'solved': len(answers) == 1,
                'sample': ' '.join(sorted(answers)[:sample]),
            })
    return (pd.DataFrame(rows)
              .sort_values(['depth', 'n_answers', 'opener', 'guesses'],
                           ascending=[True, False, True, True])
              .reset_index(drop=True))


pos = positions(TREE_OPENERS)
print(f'{LABEL}: {len(pos):,} positions over {pos["opener"].nunique()} tree(s), '
      f'depth {pos["depth"].min()}-{pos["depth"].max()}')
print(f'{pos["candidate"].sum():,} of them play a word that could itself be the answer; '
      f'the other {(~pos["candidate"]).sum():,} are probes')
display(pos.head(50))

## Positions at one depth

`DEPTH` guesses are already in, and `guess` is the next one. `DEPTH = 0` is the root — one
position, the empty history, whose move is the opener itself.

In [ ]:
DEPTH = 1      # <-- guesses already played

at = pos[pos['depth'] == DEPTH]
live = at[~at['solved']]
_cand_ans = int(at.loc[at['candidate'], 'n_answers'].sum())
print(f'depth {DEPTH} ({LABEL}): {len(at):,} positions, {len(live):,} of them with more '
      f'than one answer left, covering {at["n_answers"].sum():,} answers')
print(f'{int(at["candidate"].sum()):,} positions play a possible answer '
      f'({_cand_ans:,} answers, {_cand_ans / at["n_answers"].sum():.2%}); '
      f'the rest are probes')
display(at.head(50).reset_index(drop=True))

### ...grouped by the next guess

The same depth-`DEPTH` positions, collapsed onto the optimal next guess: `n_answers` is the
total answers that guess is played against across every position it serves, `n_patterns` is
how many distinct feedback histories lead to it, `n_openers` how many of the selected trees
play it, and `patterns` lists the histories (in the same descending-`n_answers` order as the
table above, `|`-separated once a history is more than one pattern long). Sorted by
`n_answers` descending.

With several trees pooled a guess row spans them: two trees reaching different histories and
answering with the same word land in one row, and the `patterns` strings are then per-tree —
`openers` says which. At `TOP_K = 1` both extra columns are constant.

In [ ]:
by_guess = (at.groupby('guess', sort=False)
              .agg(n_answers=('n_answers', 'sum'),
                   n_patterns=('patterns', 'size'),
                   n_openers=('opener', 'nunique'),
                   openers=('opener', lambda s: ' '.join(sorted(set(s)))),
                   patterns=('patterns', ' | '.join))
              .sort_values('n_answers', ascending=False)
              .reset_index())

print(f'depth {DEPTH} ({LABEL}): {len(by_guess):,} distinct next guesses over '
      f'{len(at):,} positions')
display(by_guess)

### Letter frequencies among those guesses

Every letter appearing in the depth-`DEPTH` next guesses, weighted by `n_answers` — a
letter's `n_answers` is the total answers sitting in the positions whose guess contains it,
so a guess played against 340 answers counts 340 times more than one played against 1.
`share` is that against all answers at this depth; it sums to well over 1, since each
position contributes its answers to five letters at once.

`LIVE_ONLY` drops positions already down to a single answer (where `guess` is just that
answer being played) so only real decisions count. `COUNT_REPEATS` decides whether a
doubled letter counts twice in its word.

Both knobs set `words`, the frame this cell and the next one work from.

In [ ]:
import string

LIVE_ONLY    = False   # <-- True: ignore positions with one answer left (guess = that answer)
COUNT_REPEATS = False  # <-- True: 'abbey' counts b twice; False: each letter once per word

words = at[~at['solved']] if LIVE_ONLY else at
_total = words['n_answers'].sum()

rows = []
for letter in string.ascii_lowercase:
    hits = words['guess'].apply(lambda w: w.count(letter) if COUNT_REPEATS else int(letter in w))
    if not hits.any():
        continue
    rows.append({'letter': letter,
                 'n_answers': int((hits * words['n_answers']).sum()),
                 'n_positions': int((hits > 0).sum()),
                 'n_guesses': words.loc[hits > 0, 'guess'].nunique()})

letter_freq = (pd.DataFrame(rows)
                 .assign(share=lambda d: d['n_answers'] / _total)
                 .sort_values('n_answers', ascending=False)
                 .reset_index(drop=True))

print(f'depth {DEPTH}: {len(letter_freq)} distinct letters over {len(words):,} positions '
      f'({_total:,} answers){" -- live positions only" if LIVE_ONLY else ""}')
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(letter_freq['letter'], letter_freq['n_answers'], color='#3b6ea5')
ax.set_ylabel('answers'); ax.set_xlabel('letter')
ax.set_title(f'Letters in the next guess at depth {DEPTH}, weighted by answers ({LABEL})')
plt.show()
display(letter_freq)

### How many letters are new

A guess's **new letters** are the distinct letters in it that none of the previous guesses
carried — how much fresh alphabet the strategy buys at this position, from 0 (every letter
already played) to 5. At `DEPTH = 0` there is no history, so it is just the opener's
distinct-letter count.

The table is the distribution over the same `words` frame: how many positions land on each
count, and how many answers they cover between them. `cand_answers` is how many of those
answers sit at a position whose guess is itself still a possible answer, and `cand_share` is
that as a fraction of the row — so a row reads "this much fresh alphabet, bought this often,
and this often by a word that could also have won outright".

The next cell takes the other side of it — the letters that are *not* new, broken down by
what the strategy already knew about each one.

In [ ]:
seen = words['guesses'].apply(lambda h: set(h.replace(' ', '')))
newness = words.assign(new_letters=[len(set(g) - h) for g, h in zip(words['guess'], seen)])
# the answers at this position, but only where the guess could itself have won
newness = newness.assign(cand=newness['n_answers'].where(newness['candidate'], 0))

new_dist = (newness.groupby('new_letters')
                   .agg(n_positions=('guess', 'size'),
                        n_answers=('n_answers', 'sum'),
                        cand_positions=('candidate', 'sum'),
                        cand_answers=('cand', 'sum'))
                   .assign(share=lambda d: d['n_answers'] / d['n_answers'].sum(),
                           cand_share=lambda d: d['cand_answers'] / d['n_answers'])
                   .reset_index())

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(new_dist['new_letters'], new_dist['n_answers'] - new_dist['cand_answers'],
       color='#3b6ea5', label='probe')
ax.bar(new_dist['new_letters'], new_dist['cand_answers'], color='#c1662f',
       bottom=new_dist['n_answers'] - new_dist['cand_answers'], label='possible answer')
ax.set_xlabel('new letters in the next guess'); ax.set_ylabel('answers')
ax.set_xticks(range(0, 6)); ax.legend()
ax.set_title(f'New letters at depth {DEPTH} ({LABEL})')
plt.show()
display(new_dist)

print('positions with the fewest new letters:')
display(newness.sort_values(['new_letters', 'n_answers'], ascending=[True, False])
               .head(15)[['opener', 'guesses', 'patterns', 'n_answers', 'guess',
                          'candidate', 'new_letters']]
               .reset_index(drop=True))

### Where the repeated letters come from

The same guesses, one TILE at a time: for each of the five tiles of the next guess, what did
the history already say about that letter, and is the tile in the **same place** it said it
about? At `DEPTH = 1` the history is just the opener, so this is "what the second word does
with the first word's letters".

| role | the tile's letter... |
| --- | --- |
| `new` | appeared in no previous guess |
| `green same place` | was green, and this is that square — the confirmed letter, replayed where it belongs |
| `green elsewhere` | was green, and this is a different square |
| `yellow same place` | was yellow *at this very square* — a square the letter is known **not** to occupy |
| `yellow elsewhere` | was yellow, and this is one of the squares it might occupy |
| `grey same place` | was grey at this square, and is played there again |
| `grey elsewhere` | was grey, and is played at a different square |
| `duplicate in guess` | appeared **earlier in this same guess** |

**The rules, exactly.** A letter's known state is the best the history ever gave it: green
if it came back green anywhere, else yellow, else grey (a letter can be several at once —
a doubled letter in one guess can be yellow at one square and grey at another). `same place`
means this tile's index is one where the letter got *that* colour.

`duplicate in guess` takes precedence over everything, so the categories never overlap and
the five roles of a guess always account for the whole word — a second `l` in `rally` is
charged to the doubling, not to the `l` the opener already found. That is the reading of
"the guess just doubles a letter itself"; the per-word count below reports the same thing
without the precedence question.

Counts are **answer-weighted** the same way the letter table is: a position holding 340
answers contributes 340 to each of its five tiles, so `share` is over `5 x` the answers at
this depth and sums to 1. `n_tiles`/`n_positions` are the unweighted counts — how many tiles
and how many distinct positions. A `0` is a result, not a missing row: it says the strategy
never once does that.

**Split by whether the guess is itself a possible answer.** `cand_*` counts the half of each
role played by a guess that is still compatible — a word that could win outright — against
the `probe` half, played only for what it reveals. `cand_share` is that as a fraction of the
role, so it reads "when the strategy reuses a letter this way, this often the word could
also have been the answer". The two halves are a partition of the role, and the whole
question is a property of the POSITION rather than the tile, so every tile of a given guess
falls on the same side of it.

Uses the same `words` frame as the two cells above, so `LIVE_ONLY` applies here too — with
it off, the positions already down to one answer are included, and those are forced to
replay whatever the single remaining answer contains and are candidates by construction.

In [ ]:
REPEAT_ROLES = ['new', 'green same place', 'green elsewhere',
                'yellow same place', 'yellow elsewhere',
                'grey same place', 'grey elsewhere', 'duplicate in guess']


def tile_roles(guesses, patterns, guess):
    """One role per TILE of `guess`, from REPEAT_ROLES -- see the markdown above.

    `guesses`/`patterns` are the history as lists of strings. A letter's known
    state is the BEST the history ever gave it (green, else yellow, else grey);
    `same place` means this tile's index is one where the letter got that colour.
    A tile whose letter appeared EARLIER IN THIS SAME GUESS is charged to
    `duplicate in guess` whatever the history says, so the roles partition the
    word."""
    seen = {}
    for g, p in zip(guesses, patterns):
        for i, (c, tile) in enumerate(zip(g, p)):
            seen.setdefault(c, {'g': set(), 'y': set(), 'b': set()})[tile].add(i)

    roles = []
    for i, c in enumerate(guess):
        if c in guess[:i]:
            roles.append('duplicate in guess')
            continue
        where = seen.get(c)
        if where is None:
            roles.append('new')
            continue
        for tile, name in (('g', 'green'), ('y', 'yellow'), ('b', 'grey')):
            if where[tile]:
                roles.append(f'{name} same place' if i in where[tile] else f'{name} elsewhere')
                break
    return roles


roles = [tile_roles(h.split(), p.split(), g)
         for h, p, g in zip(words['guesses'], words['patterns'], words['guess'])]
_ans = words['n_answers'].to_numpy()
_cand = words['candidate'].to_numpy()          # the guess could itself be the answer

rows = []
for role in REPEAT_ROLES:
    per_pos = np.array([r.count(role) for r in roles])
    ans_tiles = per_pos * _ans
    rows.append({'role': role,
                 'n_tiles': int(per_pos.sum()),
                 'n_positions': int((per_pos > 0).sum()),
                 'n_answer_tiles': int(ans_tiles.sum()),
                 'cand_tiles': int(per_pos[_cand].sum()),
                 'cand_positions': int((per_pos[_cand] > 0).sum()),
                 'cand_answer_tiles': int(ans_tiles[_cand].sum())})

repeat_dist = (pd.DataFrame(rows)
                 .assign(share=lambda d: d['n_answer_tiles'] / d['n_answer_tiles'].sum(),
                         cand_share=lambda d: d['cand_answer_tiles'] / d['n_answer_tiles']))

_tiles_played = int(_ans.sum()) * wordle.WORD_LEN
_cand_ans = int(_ans[_cand].sum())
print(f'depth {DEPTH} ({LABEL}): {len(words):,} positions, {_ans.sum():,} answers, '
      f'{_tiles_played:,} tiles played'
      f'{" -- live positions only" if LIVE_ONLY else ""}')
print(f'{int(_cand.sum()):,} of those positions play a possible answer '
      f'({_cand_ans:,} answers, {_cand_ans / _ans.sum():.2%}); the rest are probes')

_repeats = repeat_dist[repeat_dist['role'] != 'new']
_new_share = float(repeat_dist.loc[repeat_dist['role'] == 'new', 'share'].iloc[0])
_x = np.arange(len(_repeats))
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(_x - 0.2, (_repeats['n_answer_tiles'] - _repeats['cand_answer_tiles']) / _tiles_played,
       width=0.4, color='#3b6ea5', label='probe')
ax.bar(_x + 0.2, _repeats['cand_answer_tiles'] / _tiles_played,
       width=0.4, color='#c1662f', label='guess is a possible answer')
ax.set_xticks(_x); ax.set_xticklabels(_repeats['role'], rotation=30, ha='right')
ax.set_ylabel('share of tiles played'); ax.legend()
ax.set_title(f'What the repeated tiles repeat at depth {DEPTH} ({LABEL})'
             f' -- {_new_share:.1%} of tiles are new and not shown')
plt.show()
display(repeat_dist)

_dup = words['guess'].apply(lambda w: len(set(w)) < wordle.WORD_LEN)
_dup_ans = int(words.loc[_dup, 'n_answers'].sum())
_dup_cand = int(words.loc[_dup & words['candidate'], 'n_answers'].sum())
print(f'the guess itself carries a doubled letter in {int(_dup.sum()):,} of {len(words):,} '
      f'positions, {_dup_ans:,} of {_ans.sum():,} answers ({_dup_ans / _ans.sum():.2%})'
      + (f'; {_dup_cand:,} of those answers ({_dup_cand / _dup_ans:.2%}) at a position '
         f'whose guess could itself have won' if _dup_ans else ''))

roled = words.assign(roles=[' | '.join(r) for r in roles])
picks = []
for role in REPEAT_ROLES[1:]:
    hit = roled[roled['roles'].str.contains(role, regex=False)]
    if hit.empty:
        continue
    for candidate in (True, False):             # the biggest of each kind
        side = hit[hit['candidate'] == candidate]
        if side.empty:
            continue
        top = side.sort_values('n_answers', ascending=False).iloc[0]
        picks.append({'role': role, 'candidate': candidate, 'opener': top['opener'],
                      'guesses': top['guesses'], 'patterns': top['patterns'],
                      'guess': top['guess'], 'n_answers': top['n_answers'],
                      'roles': top['roles']})

print('the biggest position exercising each role, as a possible answer and as a probe:')
with pd.option_context('display.max_colwidth', 120):
    display(pd.DataFrame(picks))

## Look up one position

Filter by the history you have in hand — it is just a prefix of the `guesses`/`patterns`
columns. Each selected tree that reaches that history gets its own block: the position
itself, then every outcome its optimal next guess can produce from there. An empty
`HISTORY`/`FEEDBACK` is the root, so it prints one block per tree.

In [ ]:
HISTORY  = 'salet'     # <-- guesses so far, space separated ('' for the root)
FEEDBACK = 'bbbyb'     # <-- their feedback, space separated `byg` strings ('' for the root)

hits = pos[(pos['guesses'] == HISTORY) & (pos['patterns'] == FEEDBACK)]
if hits.empty:
    print(f'no position with guesses {HISTORY!r} and patterns {FEEDBACK!r} in any of '
          f'{len(TREE_OPENERS)} tree(s) ({LABEL}) -- unreachable under these strategies, '
          f'or a typo')
for _, r in hits.iterrows():
    print(f'{r["opener"]}: {r["n_answers"]:,} answers left; '
          f'optimal next guess: {r["guess"]!r}')
    display(hits[hits['opener'] == r['opener']].reset_index(drop=True))
    kids = pos[(pos['opener'] == r['opener'])
               & (pos['guesses'] == (HISTORY + ' ' + r['guess']).strip())]
    print(f'{len(kids)} outcome(s) of {r["guess"]!r} here:')
    display(kids.reset_index(drop=True))

## Cross-check the counts

`n_answers` is read off the tree's structure (lines sharing a prefix). This re-derives it
from scratch for a random sample of positions — filtering the 2315 answers by replaying
each guess/feedback pair through `wordle.pattern` — so a misparse shows up as a mismatch
rather than as a plausible-looking wrong number. Same spirit as `tools/checktrees.py`. The
sample is drawn across every selected tree at once, since a history already names the tree
it belongs to.

In [ ]:
N_CHECK = 200      # <-- how many random positions to re-derive

_answers = wordle.load_words('mystery')
_tiles = {'b': 0, 'y': 1, 'g': 2}

def answers_after(guesses, patterns):
    """The answers consistent with a history, from the feedback function alone."""
    alive = list(_answers)
    for g, p in zip(guesses, patterns):
        want = sum(_tiles[c] * 3 ** (wordle.WORD_LEN - 1 - i) for i, c in enumerate(p))
        alive = [a for a in alive if wordle.pattern(g, a) == want]
    return alive

rng = np.random.default_rng(0)                       # reproducible sample
idx = rng.choice(len(pos), size=min(N_CHECK, len(pos)), replace=False)
bad = []
for i in idx:
    r = pos.iloc[int(i)]
    n = len(answers_after(r['guesses'].split(), r['patterns'].split()))
    if n != r['n_answers']:
        bad.append((r['opener'], r['guesses'], r['patterns'], r['n_answers'], n))

print(f'checked {len(idx)} positions over {pos.iloc[idx]["opener"].nunique()} tree(s) '
      f'({LABEL}): ' + ('all counts agree' if not bad else f'{len(bad)} MISMATCH(ES)'))
for opener, g, p, tree_n, real_n in bad[:10]:
    print(f'  {opener}: {g} | {p}: table says {tree_n}, filter says {real_n}')